# EN4554 — Assignment 2 on CNNs

**Name:** `<<< YOUR NAME >>>`
**Index number:** `<<< YOUR INDEX >>>`
**GitHub:** https://github.com/ravindu374/EN4554_Assignment_2

---

### Acknowledgement of AI tool use

Claude (Anthropic) was used as a coding assistant while preparing this notebook: to scaffold the
layer classes, the gradient-checking harness and the plotting code. All implementations were
reviewed, run and verified by me, and all interpretations of the results are my own. I understand
every line of the code presented here.

---

### Contents
- **Setup** — data, splits, normalization, seeds
- **Task 1** — Backpropagation from scratch (no autograd)
- **Task 2** — Optimizers *(to come)*
- **Task 3** — Regularization and residual connections *(to come)*
- **Task 4** — Best CNN under 500k parameters *(to come)*

## Setup

Everything below is shared by all four tasks.

**Decisions made here, and why:**

1. **Seeds are fixed** (`SEED = 0`) for Python, NumPy and PyTorch so every number in this report is
   reproducible.
2. **The 50,000 training images are split 45,000 / 5,000** into train and validation using a single
   fixed permutation. The split never changes between tasks, so results are comparable.
3. **Per-channel normalization statistics are computed from the 45k training split only** — not from
   all 50k and not from the test set. Using validation or test pixels to compute the mean would leak
   information out of those sets.
4. **The test set is loaded but deliberately left untouched until Task 4**, where it is evaluated
   exactly once.
5. **The whole dataset is kept on the GPU as `uint8`.** CIFAR-10 is only 45000x3x32x32 bytes = 138 MB,
   which fits easily in 6 GB of VRAM. Batches are converted to float and normalized on the fly.
   This removes the `DataLoader` worker overhead entirely, which matters on Windows where spawning
   worker processes is expensive.

In [ ]:
import os, math, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets

# --- Windows + Python 3.10 SSL workaround -------------------------------------------
# Loading the Windows certificate store can fail with
#   ssl.SSLError: [ASN1: NOT_ENOUGH_DATA]
# if any certificate in it is malformed, which breaks the CIFAR-10 download.
# Passing an explicit CA file makes create_default_context skip the system store.
import ssl, certifi, functools
ssl._create_default_https_context = functools.partial(ssl.create_default_context,
                                                      cafile=certifi.where())

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR    = 'data'
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

torch.backends.cudnn.benchmark = True   # autotune conv algorithms (inputs are a fixed size)

print('torch      :', torch.__version__)
print('device     :', DEVICE)
if DEVICE.type == 'cuda':
    print('gpu        :', torch.cuda.get_device_name(0))
    print('vram       : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

### Helpers for saving and reloading results

Training runs are slow and the notebook has to be re-exported to PDF several times. Every experiment
therefore **writes its history to `results/*.json`**, and the plotting cells **read from those files**.
Re-running the notebook to fix a typo in a plot label does not retrain anything.

Set `FORCE_RETRAIN = True` to ignore the cache and rerun.

In [ ]:
FORCE_RETRAIN = False

def save_result(name, obj):
    with open(os.path.join(RESULTS_DIR, name + '.json'), 'w') as f:
        json.dump(obj, f, indent=1)

def load_result(name):
    path = os.path.join(RESULTS_DIR, name + '.json')
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def cached(name, fn):
    """Return the cached result for `name`, otherwise run fn(), cache it and return it."""
    r = load_result(name)
    if r is None:
        t0 = time.time()
        r = fn()
        r['_seconds'] = round(time.time() - t0, 1)
        save_result(name, r)
        print('computed %s in %.1fs' % (name, r['_seconds']))
    else:
        print('loaded cached %s (took %.1fs when run)' % (name, r.get('_seconds', -1)))
    return r

In [ ]:
# ---- Load CIFAR-10 and build the 45k / 5k / 10k splits -----------------------------

train_full = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True)
test_set   = datasets.CIFAR10(root=DATA_DIR, train=False, download=True)
CLASSES    = train_full.classes

# .data is a uint8 numpy array of shape (N, 32, 32, 3) -> we want (N, 3, 32, 32)
X_all = torch.from_numpy(train_full.data).permute(0, 3, 1, 2).contiguous()
Y_all = torch.tensor(train_full.targets)
X_te  = torch.from_numpy(test_set.data).permute(0, 3, 1, 2).contiguous()
Y_te  = torch.tensor(test_set.targets)

# One fixed permutation defines the split for the whole assignment
g    = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]

X_tr, Y_tr = X_all[tr_idx], Y_all[tr_idx]
X_va, Y_va = X_all[va_idx], Y_all[va_idx]

# Per-channel mean/std from the TRAINING SPLIT ONLY, on the [0,1] scale
_tr_float = X_tr.float().div_(255.0)
MEAN = _tr_float.mean(dim=(0, 2, 3))
STD  = _tr_float.std(dim=(0, 2, 3))
del _tr_float

print('train %s  val %s  test %s' % (tuple(X_tr.shape), tuple(X_va.shape), tuple(X_te.shape)))
print('channel mean:', [round(v, 4) for v in MEAN.tolist()])
print('channel std :', [round(v, 4) for v in STD.tolist()])

# Move everything to the GPU as uint8 (138 MB) and keep the normalizer there too
X_tr, Y_tr = X_tr.to(DEVICE), Y_tr.to(DEVICE)
X_va, Y_va = X_va.to(DEVICE), Y_va.to(DEVICE)
X_te, Y_te = X_te.to(DEVICE), Y_te.to(DEVICE)   # NOT used until Task 4
MEAN_D = MEAN.view(1, 3, 1, 1).to(DEVICE)
STD_D  = STD.view(1, 3, 1, 1).to(DEVICE)

def normalize(xb_uint8):
    """uint8 (B,3,32,32) -> normalized float32 (B,3,32,32)."""
    return (xb_uint8.float().div_(255.0) - MEAN_D) / STD_D

def iterate(X, Y, batch_size, shuffle=True, generator=None, flatten=False):
    """Yield normalized (xb, yb) batches straight from GPU memory."""
    n = X.shape[0]
    if shuffle:
        order = torch.randperm(n, device=X.device, generator=generator)
    else:
        order = torch.arange(n, device=X.device)
    for i in range(0, n, batch_size):
        idx = order[i:i + batch_size]
        xb  = normalize(X[idx])
        if flatten:
            xb = xb.reshape(xb.shape[0], -1)
        yield xb, Y[idx]

print('GPU memory held by the dataset: %.0f MB' % (torch.cuda.memory_allocated() / 1024**2))

---
# Task 1: Backpropagation

The goal is to implement the forward **and** backward pass of a small MLP by hand, with no autograd,
and then prove the gradients are right.

## The three derivatives I need

Write $g = \dfrac{\partial L}{\partial y}$ for the gradient arriving from the layer above.

### 1. Linear layer, $y = xW + b$

With $x \in \mathbb{R}^{N \times d_{in}}$, $W \in \mathbb{R}^{d_{in} \times d_{out}}$, $b \in \mathbb{R}^{d_{out}}$:

$$\frac{\partial L}{\partial x} = g W^\top, \qquad
  \frac{\partial L}{\partial W} = x^\top g, \qquad
  \frac{\partial L}{\partial b} = \sum_{n=1}^{N} g_n$$

The easy way to remember the first two: *to differentiate a matrix product, multiply by the other
factor, transposed* — and the shapes only work out one way, which is a useful check. The bias was
added to every row of the batch, so its gradient is the sum down the rows.

### 2. ReLU, $y = \max(x, 0)$

ReLU is a gate. Where the input was positive it passed the signal forward, so it passes the gradient
back; where the input was negative it blocked the signal, so it blocks the gradient:

$$\frac{\partial L}{\partial x} = g \odot \mathbb{1}[x > 0]$$

### 3. Softmax + cross-entropy, fused

Keeping these as one block is both more stable and much simpler. For logits $z \in \mathbb{R}^{N \times C}$
and true labels $y$:

$$L = \frac{1}{N}\sum_{n} \Big( \log \textstyle\sum_c e^{z_{nc}} - z_{n y_n} \Big),
\qquad
\frac{\partial L}{\partial z} = \frac{1}{N}\big( \mathrm{softmax}(z) - \text{onehot}(y) \big)$$

That gradient is just **"predicted probability minus true probability"**. If the model assigns 0.7 to
the correct class, the gradient on that logit is $-0.3$: push it up. Every messy term of the softmax
Jacobian cancels against the $\log$ in the cross-entropy — which is exactly why the two are always
fused in practice.

**Numerical stability.** $e^{z}$ overflows to `inf` for logits above about 88 in float32, which turns
the loss into `nan`. Subtracting the row-wise maximum before exponentiating leaves the softmax
unchanged (it cancels top and bottom) but keeps every exponent $\le 0$.

## (a) Linear, ReLU and softmax cross-entropy with explicit forward/backward

Each layer stores whatever its backward pass will need during `forward`. Parameters and their
gradients are exposed through `params()` / `grads()` so the optimizer and the gradient checker can
find them generically.

In [ ]:
class Linear:
    """y = x @ W + b, with hand-written backward."""

    def __init__(self, n_in, n_out, init='kaiming', dtype=torch.float32,
                 device=DEVICE, generator=None):
        if init == 'kaiming':     # for ReLU: Var[W] = 2 / fan_in
            std = math.sqrt(2.0 / n_in)
        elif init == 'xavier':    # for sigmoid/tanh: Var[W] = 1 / fan_in
            std = math.sqrt(1.0 / n_in)
        else:
            raise ValueError(init)

        self.W = torch.randn(n_in, n_out, generator=generator, dtype=dtype, device=device) * std
        self.b = torch.zeros(n_out, dtype=dtype, device=device)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)
        self.x  = None

    def forward(self, x):
        self.x = x                      # needed for dL/dW
        return x @ self.W + self.b

    def backward(self, g):
        self.dW = self.x.T @ g          # (in, N) @ (N, out) -> (in, out)
        self.db = g.sum(dim=0)          # bias is broadcast over the batch
        return g @ self.W.T             # (N, out) @ (out, in) -> (N, in)

    def params(self): return {'W': self.W,  'b': self.b}
    def grads(self):  return {'W': self.dW, 'b': self.db}


class ReLU:
    def forward(self, x):
        self.mask = (x > 0)             # cheaper to store than x itself
        return x * self.mask

    def backward(self, g):
        return g * self.mask

    def params(self): return {}
    def grads(self):  return {}


class Sigmoid:
    """Only needed for Task 1(d)."""
    def forward(self, x):
        self.y = torch.sigmoid(x)       # store the OUTPUT: y' = y(1-y)
        return self.y

    def backward(self, g):
        return g * self.y * (1.0 - self.y)

    def params(self): return {}
    def grads(self):  return {}


class SoftmaxCrossEntropy:
    """Fused softmax + cross-entropy, averaged over the batch."""

    def forward(self, z, y):
        zs       = z - z.max(dim=1, keepdim=True).values      # stability shift
        exp      = zs.exp()
        sum_exp  = exp.sum(dim=1, keepdim=True)
        self.p   = exp / sum_exp                              # softmax probabilities
        self.y   = y
        self.N   = z.shape[0]
        rows     = torch.arange(self.N, device=z.device)
        log_p_correct = zs[rows, y] - sum_exp.squeeze(1).log()
        return -log_p_correct.mean()

    def backward(self):
        g = self.p.clone()
        g[torch.arange(self.N, device=g.device), self.y] -= 1.0
        return g / self.N

In [ ]:
class MLP:
    """sizes=[3072, 512, 256, 10] -> Linear/act/Linear/act/Linear."""

    def __init__(self, sizes, act='relu', init=None, dtype=torch.float32,
                 device=DEVICE, generator=None):
        if init is None:
            init = 'kaiming' if act == 'relu' else 'xavier'
        Act = {'relu': ReLU, 'sigmoid': Sigmoid}[act]

        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1], init=init, dtype=dtype,
                                      device=device, generator=generator))
            if i < len(sizes) - 2:                 # no activation after the output layer
                self.layers.append(Act())

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, g):
        for layer in reversed(self.layers):        # the whole point: reverse order
            g = layer.backward(g)
        return g

    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def param_list(self):
        """[(layer, name)] pairs, for the optimizer and the gradient checker."""
        out = []
        for l in self.linears():
            out.append((l, 'W'))
            out.append((l, 'b'))
        return out

    def n_params(self):
        return sum(l.W.numel() + l.b.numel() for l in self.linears())


class SGDManual:
    """Plain SGD written out, so Task 1 uses no torch.optim either."""

    def __init__(self, model, lr):
        self.model, self.lr = model, lr

    @torch.no_grad()
    def step(self):
        for layer, name in self.model.param_list():
            layer.params()[name] -= self.lr * layer.grads()[name]


gen  = torch.Generator(device=DEVICE).manual_seed(SEED)
demo = MLP([3072, 512, 256, 10], act='relu', generator=gen)
print('MLP 3072-512-256-10 has %s parameters' % format(demo.n_params(), ','))

## (b) Verifying the gradients

Two checks, which fail for different reasons — so passing both is strong evidence.

**Check 1 — against autograd.** The same network is rebuilt with `nn.Linear`, my weights are copied
in, and PyTorch's gradients are compared with mine using `torch.allclose`. One detail matters:
`nn.Linear` stores its weight as `(out_features, in_features)` while mine is `(in, out)`, so the
copy and the comparison both need a **transpose**.

**Check 2 — centred finite differences.** This trusts nothing at all. For a single parameter
$\theta$:

$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \varepsilon) - L(\theta - \varepsilon)}{2\varepsilon}$$

The *centred* version is used rather than the one-sided one because its error is
$\mathcal{O}(\varepsilon^2)$ instead of $\mathcal{O}(\varepsilon)$. The relative error reported is

$$\text{rel} = \frac{|\text{analytic} - \text{numeric}|}{|\text{analytic}| + |\text{numeric}| + 10^{-12}}$$

**This check is run in `float64`.** In float32 the subtraction $L(\theta+\varepsilon)-L(\theta-\varepsilon)$
cancels most of the significant digits and the relative error floors out around $10^{-3}$, which looks
like a bug but is only rounding. In float64 a correct implementation reaches about $10^{-10}$.

In [ ]:
# ---- Check 1: against autograd ------------------------------------------------------

def check_against_autograd(sizes=(256, 64, 32, 10), N=16, seed=0):
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    model = MLP(list(sizes), act='relu', dtype=torch.float64, device=DEVICE, generator=gen)
    crit  = SoftmaxCrossEntropy()

    x = torch.randn(N, sizes[0], dtype=torch.float64, device=DEVICE, generator=gen)
    y = torch.randint(0, sizes[-1], (N,), device=DEVICE, generator=gen)

    # --- mine
    loss_mine = crit.forward(model.forward(x), y)
    model.backward(crit.backward())

    # --- PyTorch's, same weights
    mods = []
    for i, l in enumerate(model.linears()):
        m = nn.Linear(l.W.shape[0], l.W.shape[1]).to(DEVICE).double()
        m.weight.data.copy_(l.W.T)      # nn.Linear is (out, in); mine is (in, out)
        m.bias.data.copy_(l.b)
        mods.append(m)
        if i < len(model.linears()) - 1:
            mods.append(nn.ReLU())
    ref = nn.Sequential(*mods)

    loss_ref = F.cross_entropy(ref(x), y)
    loss_ref.backward()

    print('loss  mine = %.12f   autograd = %.12f   allclose = %s'
          % (loss_mine.item(), loss_ref.item(), torch.allclose(loss_mine, loss_ref)))
    print()
    print('%-8s %-14s %-14s %s' % ('layer', 'max|dW diff|', 'max|db diff|', 'allclose(W, b)'))
    all_ok = True
    for i, l in enumerate(model.linears()):
        dW_ref = mods[2 * i].weight.grad.T
        db_ref = mods[2 * i].bias.grad
        okW = torch.allclose(l.dW, dW_ref, rtol=1e-10, atol=1e-12)
        okb = torch.allclose(l.db, db_ref, rtol=1e-10, atol=1e-12)
        all_ok = all_ok and okW and okb
        print('%-8d %-14.3e %-14.3e %s, %s'
              % (i + 1, (l.dW - dW_ref).abs().max().item(),
                 (l.db - db_ref).abs().max().item(), okW, okb))
    print()
    print('ALL GRADIENTS MATCH AUTOGRAD:', all_ok)
    return all_ok

_ = check_against_autograd()

In [ ]:
# ---- Check 2: centred finite differences --------------------------------------------

def check_finite_differences(sizes=(256, 64, 32, 10), N=8, eps=1e-5,
                             n_per_tensor=12, seed=1, verbose=True):
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    model = MLP(list(sizes), act='relu', dtype=torch.float64, device=DEVICE, generator=gen)
    crit  = SoftmaxCrossEntropy()

    x = torch.randn(N, sizes[0], dtype=torch.float64, device=DEVICE, generator=gen)
    y = torch.randint(0, sizes[-1], (N,), device=DEVICE, generator=gen)

    def loss_now():
        return crit.forward(model.forward(x), y).item()

    # analytic gradients once, then cloned so later forwards cannot disturb them
    crit.forward(model.forward(x), y)
    model.backward(crit.backward())
    analytic = {(i, nm): model.linears()[i].grads()[nm].clone()
                for i in range(len(model.linears())) for nm in ('W', 'b')}

    rows = []
    rng  = np.random.default_rng(seed)
    for i, layer in enumerate(model.linears()):
        for nm in ('W', 'b'):
            P = layer.params()[nm].view(-1)
            G = analytic[(i, nm)].view(-1)
            errs = []
            for k in rng.choice(P.numel(), size=min(n_per_tensor, P.numel()), replace=False):
                k = int(k)
                orig = P[k].item()
                P[k] = orig + eps; lp = loss_now()
                P[k] = orig - eps; lm = loss_now()
                P[k] = orig                                   # restore exactly
                numeric = (lp - lm) / (2 * eps)
                ana     = G[k].item()
                errs.append(abs(ana - numeric) / (abs(ana) + abs(numeric) + 1e-12))
            rows.append(('layer %d  %s' % (i + 1, nm), float(np.max(errs)), float(np.median(errs))))

    if verbose:
        print('centred finite differences, float64, eps = %g, %d params per tensor'
              % (eps, n_per_tensor))
        print()
        print('%-16s %-14s %-14s %s' % ('tensor', 'max rel err', 'median rel err', 'pass (<1e-7)'))
        for name, mx, med in rows:
            print('%-16s %-14.3e %-14.3e %s' % (name, mx, med, mx < 1e-7))
        print()
        print('WORST relative error overall: %.3e' % max(r[1] for r in rows))
    return rows

fd_rows = check_finite_differences()
save_result('t1b_finite_diff', {'rows': fd_rows})

### Interpretation — Task 1(b)

*(Fill the blanks from the output above, then keep or trim the wording.)*

Both checks pass. Every layer's `dW` and `db` agree with autograd to within `___` in absolute terms,
and `torch.allclose` returns `True` for all of them, so the analytic backward pass is implementing
exactly the same function PyTorch does.

The finite-difference check is the stronger evidence, because it never touches autograd: the worst
relative error over all sampled parameters is `___`, close to the $10^{-10}$-ish floor set by float64
round-off. That is what a correct gradient looks like — an error of $10^{-3}$ or worse would mean a
real bug.

Two practical points worth recording:

- **Precision matters more than the formula.** Running the identical check in float32 gives relative
  errors around $10^{-3}$, because $L(\theta+\varepsilon)$ and $L(\theta-\varepsilon)$ agree in their
  first several digits and subtracting them destroys the rest. The gradient is not wrong; the
  measuring instrument is. Hence float64 and $\varepsilon = 10^{-5}$.
- **ReLU's kink can produce a false failure.** ReLU is not differentiable at 0. If a pre-activation
  happens to sit within $\varepsilon$ of zero, the $+\varepsilon$ and $-\varepsilon$ evaluations land on
  opposite sides of the kink, the two-sided slope measures something the true derivative never
  claimed, and that one parameter shows a large error even though the code is correct. With random
  weights and only `___` sampled parameters this did not occur here, but it is why a single large
  relative error in a gradient check should be investigated rather than trusted blindly.

## (c) Training the MLP on flattened CIFAR-10

Each 32x32x3 image is flattened to a **3072-dimensional vector**, and the network is
3072 -> 512 -> 256 -> 10 trained with the hand-written SGD above. No autograd is used in this cell
either — `model.backward(...)` is my own code.

In [ ]:
@torch.no_grad()
def evaluate_mlp(model, X, Y, batch_size=1000):
    correct = 0
    for xb, yb in iterate(X, Y, batch_size, shuffle=False, flatten=True):
        correct += (model.forward(xb).argmax(dim=1) == yb).sum().item()
    return correct / X.shape[0]


def train_task1c(sizes=(3072, 512, 256, 10), epochs=30, lr=0.1, batch_size=128):
    gen   = torch.Generator(device=DEVICE).manual_seed(SEED)
    model = MLP(list(sizes), act='relu', generator=gen)
    crit  = SoftmaxCrossEntropy()
    opt   = SGDManual(model, lr=lr)

    hist = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_acc': [],
            'lr': lr, 'sizes': list(sizes), 'batch_size': batch_size,
            'n_params': model.n_params()}

    for ep in range(1, epochs + 1):
        running, nb, correct, seen = 0.0, 0, 0, 0
        for xb, yb in iterate(X_tr, Y_tr, batch_size, shuffle=True,
                              generator=gen, flatten=True):
            logits = model.forward(xb)
            loss   = crit.forward(logits, yb)
            model.backward(crit.backward())        # <- hand-written backprop
            opt.step()

            running += loss.item(); nb += 1
            correct += (logits.argmax(dim=1) == yb).sum().item(); seen += yb.numel()

        va = evaluate_mlp(model, X_va, Y_va)
        hist['epoch'].append(ep)
        hist['train_loss'].append(running / nb)
        hist['train_acc'].append(correct / seen)
        hist['val_acc'].append(va)
        if ep == 1 or ep % 5 == 0:
            print('epoch %2d  train loss %.4f  train acc %.4f  val acc %.4f'
                  % (ep, running / nb, correct / seen, va))

    hist['final_val_acc'] = hist['val_acc'][-1]
    hist['best_val_acc']  = max(hist['val_acc'])
    return hist

t1c = cached('t1c_mlp_sgd', train_task1c)
print()
print('parameters      : %s' % format(t1c['n_params'], ','))
print('final val acc   : %.4f' % t1c['final_val_acc'])
print('best  val acc   : %.4f' % t1c['best_val_acc'])
print('train/val gap   : %.4f' % (t1c['train_acc'][-1] - t1c['final_val_acc']))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(t1c['epoch'], t1c['train_loss'], color='#2563eb')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('training loss')
ax[0].set_title('Task 1(c): training loss, hand-written backprop + SGD')
ax[0].grid(alpha=.3)

ax[1].plot(t1c['epoch'], [a * 100 for a in t1c['train_acc']], label='train', color='#2563eb')
ax[1].plot(t1c['epoch'], [a * 100 for a in t1c['val_acc']],   label='validation', color='#dc2626')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy (%)')
ax[1].set_title('Accuracy (final val = %.2f%%)' % (100 * t1c['final_val_acc']))
ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

### Interpretation — Task 1(c)

*(Fill the blanks from your run.)*

The loss falls smoothly from about `___` — close to $\ln 10 = 2.303$, which is exactly what a network
that knows nothing must output — down to `___`, and validation accuracy settles at **`___`%**.
Training accuracy ends `___` points above validation accuracy, so the network has begun to memorise
the training split; with ~1.7 M parameters against 45k images that is expected, and no regularization
is used here.

**The number to pay attention to is the ceiling.** A fully-connected network tops out at roughly
50-55% on CIFAR-10 no matter how long it trains or how wide it is made, and that is a structural
limit rather than an optimization failure:

- **Flattening destroys locality.** After `reshape`, the pixel at (5, 5) and the pixel at (5, 6) are
  just input 165 and input 166 — the network is never told they are neighbours, and would have to
  learn that fact separately for every pair of pixels.
- **There is no translation invariance.** A cat in the top-left and the same cat shifted three pixels
  right are two completely different input vectors, so the MLP must learn a separate set of weights
  for every position an object can appear in.
- **It is extremely expensive in parameters.** The first layer alone is 3072x512 = 1.57 M weights,
  and it still only ever sees one fixed arrangement of the image.

This is precisely the gap convolution closes: weight sharing gives translation invariance for free,
and a local kernel builds in the assumption that nearby pixels belong together. The small CNN in
Task 2 reaches a much higher accuracy with roughly an order of magnitude fewer parameters, which is
the comparison that motivates the rest of this assignment.

## (d) Gradient norms at initialization: sigmoid vs ReLU, 10 layers

A 10-layer MLP is built, **one** forward and backward pass is run at initialization (no training at
all), and $\lVert \partial L/\partial W^{(l)} \rVert_2$ is recorded for every layer.

Each activation is paired with the initialization it was designed for — **Xavier**
($\mathrm{Var}[W]=1/\text{fan-in}$) for sigmoid, **Kaiming** ($\mathrm{Var}[W]=2/\text{fan-in}$) for
ReLU. Pairing them this way is the honest comparison: the activation and the initializer are a
matched pair, and running sigmoid with Kaiming init would blame the activation for the wrong
initializer.

The y-axis is **logarithmic**, otherwise the sigmoid curve is an indistinguishable flat line at zero.

In [ ]:
def layer_grad_norms(act, depth=10, width=128, N=256, seed=0):
    gen   = torch.Generator(device=DEVICE).manual_seed(seed)
    sizes = [3072] + [width] * (depth - 1) + [10]
    model = MLP(sizes, act=act, generator=gen)       # init defaults to the matching one
    crit  = SoftmaxCrossEntropy()

    xb = normalize(X_tr[:N]).reshape(N, -1)
    yb = Y_tr[:N]

    crit.forward(model.forward(xb), yb)
    model.backward(crit.backward())                  # one backward pass, no update
    return [l.dW.norm().item() for l in model.linears()]


def run_task1d():
    return {'sigmoid': layer_grad_norms('sigmoid'),
            'relu':    layer_grad_norms('relu'),
            'depth': 10, 'width': 128}

t1d = cached('t1d_grad_norms', run_task1d)
print()
print('%-7s %-16s %-16s' % ('layer', 'sigmoid', 'ReLU'))
for i, (s, r) in enumerate(zip(t1d['sigmoid'], t1d['relu']), start=1):
    print('%-7d %-16.3e %-16.3e' % (i, s, r))

print()
print('last layer / first layer  --  sigmoid: %.3e    ReLU: %.3e'
      % (t1d['sigmoid'][-1] / max(t1d['sigmoid'][0], 1e-300),
         t1d['relu'][-1]    / max(t1d['relu'][0],    1e-300)))

In [ ]:
layers = list(range(1, t1d['depth'] + 1))
plt.figure(figsize=(7.5, 5))
plt.semilogy(layers, t1d['sigmoid'], 'o-', color='#dc2626', label='sigmoid + Xavier init')
plt.semilogy(layers, t1d['relu'],    's-', color='#2563eb', label='ReLU + Kaiming init')
plt.xlabel('layer index  (1 = input side, 10 = output side)')
plt.ylabel(r'$\|\partial L / \partial W^{(l)}\|_2$   (log scale)')
plt.title('Task 1(d): per-layer gradient norm at initialization, 10-layer MLP')
plt.xticks(layers); plt.grid(alpha=.3, which='both'); plt.legend()
plt.tight_layout(); plt.show()

### Interpretation — Task 1(d)

*(Fill the blanks from the table above.)*

**Sigmoid: the gradient vanishes.** The norms fall by about `___` orders of magnitude from layer 10
down to layer 1, and on the log axis that fall is close to a straight line — meaning each layer
multiplies the gradient by roughly the same constant factor. The reason is the derivative itself:

$$\sigma'(x) = \sigma(x)\,(1 - \sigma(x)) \le 0.25$$

with equality only at $x = 0$. Backpropagation multiplies by one such factor per layer, so after 9
hidden layers the signal reaching the first layer is scaled by at most
$0.25^9 \approx 4\times10^{-6}$, and in practice much less, because most units sit away from $x = 0$
where $\sigma'$ is far below 0.25. The early layers therefore receive essentially no learning signal:
they stay near their random initialization while only the last few layers train. This is the classic
**vanishing gradient problem**.

**ReLU: the gradient stays roughly flat.** The norms vary by only about `___` across all 10 layers,
because two effects cancel:

- ReLU's derivative is **0 or 1** — it never shrinks what passes through. But roughly half the units
  are inactive at initialization, so about half the signal is killed, costing a factor of about 2 in
  variance per layer.
- **Kaiming initialization puts that factor back deliberately.** Setting
  $\mathrm{Var}[W] = 2/\text{fan-in}$ instead of $1/\text{fan-in}$ is precisely the correction for
  "half the units are off". The 2 in the numerator *is* the ReLU correction.

So the activation and the initializer are not independent choices: Kaiming init is derived *for*
ReLU, and pairing Xavier with ReLU reintroduces a slow decay.

**Why this matters for the rest of the assignment.** This single plot explains why deep networks were
impractical before roughly 2010, and why ReLU together with principled initialization was what made
depth usable. It is also only a partial fix: ReLU keeps gradients healthy *at initialization*, but at
56 layers (Task 3b) degradation returns during training, and that is the problem residual connections
exist to solve — a shortcut gives the gradient a path with derivative exactly 1 all the way back to
the input.

---
## Task 1 — summary

| Part | What was done | Result |
|---|---|---|
| (a) | `Linear`, `ReLU`, `Sigmoid` and a fused `SoftmaxCrossEntropy`, each with explicit forward/backward; `MLP` and `SGDManual` built on top | no autograd used anywhere |
| (b) | Compared against autograd (`torch.allclose`) and against centred finite differences in float64 | worst relative error `___` |
| (c) | 3072-512-256-10 trained with hand-written SGD on flattened CIFAR-10 | validation accuracy `___`% |
| (d) | Per-layer gradient norms at init, 10 layers, sigmoid vs ReLU | sigmoid decays `___` orders; ReLU flat |

**Next:** Task 2 moves to `torch.optim.Optimizer` subclasses (SGD+momentum and Adam) and a small CNN.